# PlayLogic GRPO Colab

Run these cells in order on a Colab GPU runtime. The first training run is intentionally tiny so the notebook proves the environment, imports, model load, reward function, and trainer wiring before you scale it up.

In [ ]:
# 1. Clone the repo and install dependencies
import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/arjune4dev/PlayLogic.git"
REPO_DIR = pathlib.Path("/content/PlayLogic")

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-e",
    ".",
    "trl>=0.25.0",
    "transformers>=4.56.0",
    "datasets",
    "accelerate",
    "peft",
])

print("Repo ready at", pathlib.Path.cwd())

In [ ]:
# 2. Imports and environment smoke test
import json
import random
import re

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

from playlogic.models import Action
from playlogic.server.environment import MultiAgentFootball

env = MultiAgentFootball()
obs = env.reset()
actions = {i: "MOVE 0 0" for i in range(22)}
obs, rewards, done, info = env.step(Action(text=json.dumps(actions)))

print("Environment OK")
print("Rewards:", rewards)
print("Done:", done)

In [ ]:
# 3. Build a small prompt dataset from live football observations
def build_prompt(local_obs, team_name):
    return (
        f"You are a {team_name} football player.\n"
        f"Observation: {local_obs}\n"
        "Return exactly one action in this format: MOVE dx dy.\n"
        "Use dx and dy between -8 and 8.\n"
        "Action:"
    )

def collect_prompts(num_resets=4):
    prompts = []
    env = MultiAgentFootball()
    for _ in range(num_resets):
        obs = env.reset()
        obs_dict = json.loads(obs.text)
        for pid, local_obs in obs_dict.items():
            team_name = "Team A" if int(pid) < 11 else "Team B"
            prompts.append(build_prompt(local_obs, team_name))
    random.shuffle(prompts)
    return prompts

prompts = collect_prompts(num_resets=4)
dataset = Dataset.from_dict({"prompt": prompts[:32]})

print(dataset)
print(dataset[0]["prompt"][:500])

In [ ]:
# 4. Reward function for TRL GRPOTrainer
MOVE_RE = re.compile(r"\bMOVE\s+([-+]?\d*\.?\d+)\s+([-+]?\d*\.?\d+)\b", re.IGNORECASE)

def football_action_reward(completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion.strip()
        match = MOVE_RE.search(text)

        if not match:
            rewards.append(-1.0)
            continue

        dx = float(match.group(1))
        dy = float(match.group(2))
        score = 1.0

        score += 0.5 if -8.0 <= dx <= 8.0 else -0.75
        score += 0.5 if -8.0 <= dy <= 8.0 else -0.75
        score += 0.25 if abs(dx) + abs(dy) > 0.25 else -0.25

        extra_text = MOVE_RE.sub("", text).strip()
        if extra_text:
            score -= 0.25

        rewards.append(score)

    return rewards

print(football_action_reward(["MOVE 2 -1", "hello", "MOVE 99 0"]))

In [ ]:
# 5. Load a small open model and run a tiny GRPO training job
# Increase max_steps and dataset size after this cell succeeds.
model_name = "Qwen/Qwen2-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch_dtype,
    device_map="auto",
)

training_args = GRPOConfig(
    output_dir="./grpo_football",
    max_steps=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=5e-6,
    logging_steps=1,
    num_generations=2,
    max_prompt_length=256,
    max_completion_length=16,
    report_to=[],
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=football_action_reward,
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
# 6. Plot rewards and save the trained checkpoint
logs = trainer.state.log_history
reward_values = [log["reward"] for log in logs if "reward" in log]

if reward_values:
    plt.plot(reward_values)
    plt.title("GRPO Reward")
    plt.xlabel("Log step")
    plt.ylabel("Reward")
    plt.grid(True)
    plt.savefig("reward_plot.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No reward values were logged yet.")

trainer.save_model("./grpo_football/final")
print("Saved model to ./grpo_football/final")